# 🎯 Image Captioning - Exercise Solutions
## INFO 7390

---

In [ ]:
!pip install -q transformers torch torchvision pillow requests sentence-transformers

In [ ]:
import torch
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import requests
from io import BytesIO
import numpy as np
import time

device = "cuda" if torch.cuda.is_available() else "cpu"
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)

def load_image(url):
    response = requests.get(url, timeout=10)
    return Image.open(BytesIO(response.content)).convert("RGB")

TEST_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/43/Cute_dog.jpg/1200px-Cute_dog.jpg"
print("Setup complete!")

## Exercise 1 Solution

In [ ]:
def simple_caption(image_url: str) -> str:
    image = load_image(image_url)
    inputs = processor(image, return_tensors="pt").to(device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_length=50)
    return processor.decode(output_ids[0], skip_special_tokens=True)

print(simple_caption(TEST_URL))

## Exercise 2 Solution

In [ ]:
def conditional_caption(image_url: str, prompt: str = None) -> str:
    image = load_image(image_url)
    if prompt:
        inputs = processor(image, prompt, return_tensors="pt").to(device)
    else:
        inputs = processor(image, return_tensors="pt").to(device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_length=50)
    return processor.decode(output_ids[0], skip_special_tokens=True)

for p in [None, "a photo of", "this is"]:
    print(f"{p} -> {conditional_caption(TEST_URL, p)}")

## Exercise 3 Solution

In [ ]:
def benchmark_beams(image_url: str, beam_sizes=[1, 3, 5, 10]):
    results = []
    image = load_image(image_url)
    inputs = processor(image, return_tensors="pt").to(device)
    
    for beams in beam_sizes:
        start = time.time()
        with torch.no_grad():
            output_ids = model.generate(**inputs, max_length=30, num_beams=beams)
        elapsed_ms = (time.time() - start) * 1000
        caption = processor.decode(output_ids[0], skip_special_tokens=True)
        results.append({'beams': beams, 'caption': caption, 'time_ms': elapsed_ms, 'length': len(caption.split())})
    return results

for r in benchmark_beams(TEST_URL):
    print(f"Beams={r['beams']}: {r['caption']} ({r['time_ms']:.0f}ms)")

## Exercise 4 Solution

In [ ]:
def analyze_quality(image: Image.Image) -> dict:
    issues = []
    width, height = image.size
    if width < 100 or height < 100:
        issues.append(f"Low resolution: {width}x{height}")
    
    aspect_ratio = max(width, height) / min(width, height)
    if aspect_ratio > 3:
        issues.append(f"Extreme aspect ratio: {aspect_ratio:.1f}:1")
    
    img_array = np.array(image)
    brightness = np.mean(img_array)
    if brightness < 30:
        issues.append(f"Too dark: {brightness:.0f}")
    elif brightness > 240:
        issues.append(f"Too bright: {brightness:.0f}")
    
    return {'resolution': (width, height), 'aspect_ratio': aspect_ratio, 'brightness': brightness, 'issues': issues, 'is_valid': len(issues) == 0}

print(analyze_quality(load_image(TEST_URL)))

## Exercise 5 Solution

In [ ]:
class CaptionPipeline:
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
        self.model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(self.device)
    
    def caption(self, image_url: str, **kwargs) -> dict:
        start = time.time()
        image = load_image(image_url)
        inputs = self.processor(image, return_tensors="pt").to(self.device)
        
        with torch.no_grad():
            output_ids = self.model.generate(**inputs, max_length=kwargs.get('max_length', 50), num_beams=kwargs.get('num_beams', 5))
        
        caption = self.processor.decode(output_ids[0], skip_special_tokens=True)
        return {'caption': caption, 'quality': analyze_quality(image), 'time_ms': (time.time() - start) * 1000}
    
    def batch_caption(self, urls: list) -> list:
        return [self.caption(url) for url in urls]

pipeline = CaptionPipeline()
print(pipeline.caption(TEST_URL))

## Bonus Solution

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

sent_model = SentenceTransformer('all-MiniLM-L6-v2')

def estimate_confidence(image_url: str, n_samples: int = 5) -> dict:
    image = load_image(image_url)
    inputs = processor(image, return_tensors="pt").to(device)
    
    captions = []
    for _ in range(n_samples):
        with torch.no_grad():
            output = model.generate(**inputs, max_length=30, do_sample=True, temperature=0.8, top_p=0.9)
        captions.append(processor.decode(output[0], skip_special_tokens=True))
    
    embeddings = sent_model.encode(captions)
    sims = []
    for i in range(len(captions)):
        for j in range(i+1, len(captions)):
            sims.append(cosine_similarity([embeddings[i]], [embeddings[j]])[0][0])
    
    score = np.mean(sims) if sims else 0
    return {'captions': captions, 'score': score, 'confident': score > 0.8}

result = estimate_confidence(TEST_URL)
print(f"Score: {result['score']:.3f}")
for c in result['captions']:
    print(f"  - {c}")